In [4]:
def get_recurrence(data: list):
    stats = dict()
    for i in range(len(data)-1):
        pair = (data[i], data[i+1])
        stats[pair] = stats.get(pair,0) + 1
    return stats
with open('/Users/toprak/Documents/yz50/8. Hafta/nutuk.txt') as f:
    data = list(f.read().encode())

stats = sorted(((v,k) for k,v in get_recurrence(data).items()), reverse=True)
data_merged = list(data)
iters = 20
i=0
merges = {}
while i<iters:
    for j in range(len(data_merged)-1):
        if j+1>=len(data_merged): continue
        if data_merged[j] == stats[0][1][0] and data_merged[j+1] == stats[0][1][1]:
            data_merged.pop(j); data_merged.pop(j)
            data_merged.insert(j, 256+i)
    print(f'{stats[0][1][0]} ({chr(stats[0][1][0]) if stats[0][1][0] < 256 else ""}) ve {stats[0][1][1]} ({chr(stats[0][1][1]) if stats[0][1][1] < 256 else ""}) birleştirildi. Yeni token: {256+i}')
    stats = sorted(((v,k) for k,v in get_recurrence(data_merged).items()), reverse=True)
    merges[(stats[0][1][0], stats[0][1][1])] = 256+i
    i+=1
print(merges)
print(f"Küçülme oranı: %{(1 - (len(data_merged) / len(data)))*100}")

KeyboardInterrupt: 

In [5]:
def encode(data_input:str):
    data_input = data_input.encode(encoding='utf-8', errors='replace')
    skipping = False
    encoded = []
    for i in range(len(data_input)):
        pair = (data_input[i], data_input[i+1]) if i<len(data_input)-1 else (data_input[i],)
        if pair in merges and not skipping: encoded.append(merges[pair]); skipping = True
        elif skipping: skipping = False
        else: encoded.append(pair[0])
    return encoded

vocab_merges = {v: k for k, v in merges.items()}
def decode(data_input: list) -> str:
    tokens = list(data_input)
    while any(t > 255 for t in tokens):
        new_tokens = []
        for t in tokens:
            if t > 255 and t in vocab_merges:
                new_tokens.extend(vocab_merges[t])
            else:
                new_tokens.append(t)
        tokens = new_tokens

    return bytes(tokens).decode('utf-8', errors='replace')

In [7]:
decode(encode('Deneme yapıyorum 😀'))

'Deneme yapıyorum 😀'